In [2]:
#!/usr/bin/env python

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"

# === 1. Point to ONE experiment CSV ===
csv_path = os.path.join(
    OUT_ROOT,
    "results",
    "FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv",
)

print(f"Loading: {csv_path}")
df = pd.read_csv(csv_path)

# === 2. Get case START times (for warm-up trimming) ===
df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
df_start = df_start.rename(columns={"timestamp": "start_time"})

# === 3. Completed END events ===
df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
if df_complete.empty:
    raise ValueError("No COMPLETE END events found – check the log or filters.")

# Attach start_time to each completed case
df_complete = df_complete.merge(df_start, on="case_id", how="left")

# === 4. Warm-up trimming (e.g. first 10% of simulated time) ===
t_max = df["timestamp"].max()
warmup_threshold = 0.17 * t_max      # 10% of horizon

df_steady = df_complete[df_complete["start_time"] >= warmup_threshold].copy()

if df_steady.empty:
    raise ValueError("No cases left after warm-up trimming – threshold too strict?")

print(f"Total completed cases: {len(df_complete)}")
print(f"Cases after warm-up trimming: {len(df_steady)}")
print(f"Warm-up threshold (time): {warmup_threshold:.2f}")

# === 5. Var_total on trimmed cases ===
cycle_times = df_steady["cycle_time"].to_numpy(dtype=float)

mu = cycle_times.mean()
var_total = ((cycle_times - mu) ** 2).mean()   # population variance

print(f"\nMean cycle time (steady): {mu:.4f}")
print(f"Var_total (steady, population): {var_total:.4f}")

print("\nQuick summary of cycle times (steady):")
print(df_steady["cycle_time"].describe())


Loading: out/251110\results\FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv
Total completed cases: 8425
Cases after warm-up trimming: 6979
Warm-up threshold (time): 5099.99

Mean cycle time (steady): 69.4456
Var_total (steady, population): 1068.8277

Quick summary of cycle times (steady):
count    6979.000000
mean       69.445600
std        32.695274
min         6.710983
25%        45.869595
50%        64.367474
75%        88.043881
max       282.312239
Name: cycle_time, dtype: float64


In [5]:
#!/usr/bin/env python

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"
RESULTS_DIR = os.path.join(OUT_ROOT, "results")

l_value       = 0.28
styles        = ["pooled", "hybrid30", "dedicated"]
count_key     = "C1"
variant_count = 18
activity_total = 8
qc_level      = 0.97
hets          = ["identical", "mild_all"]

# Same warm-up fraction
WARMUP_FRAC = 0.17


def pop_var(x: pd.Series) -> float:
    x = x.to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    mu = x.mean()
    return ((x - mu) ** 2).mean()


def compute_descriptors_for_file(csv_path: str, warmup_frac: float = 0.17) -> dict:
    print(f"\n=== Processing {os.path.basename(csv_path)} ===")
    df = pd.read_csv(csv_path)

    # -------- warm-up selection on cases (KEEP THIS) --------
    df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
    df_start = df_start.rename(columns={"timestamp": "start_time"})

    df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
    if df_complete.empty:
        raise ValueError(f"No COMPLETE END events in {csv_path}")

    df_complete = df_complete.merge(df_start, on="case_id", how="left")

    t_max = df["timestamp"].max()
    warmup_threshold = warmup_frac * t_max
    df_steady_complete = df_complete[df_complete["start_time"] >= warmup_threshold].copy()
    if df_steady_complete.empty:
        raise ValueError(f"No cases left after warm-up in {csv_path}")

    steady_case_ids = df_steady_complete["case_id"].unique()
    df_steady = df[df["case_id"].isin(steady_case_ids)].copy()

    # -------- overall cycle-time stats (KEEP THIS) --------
    cycle_times = df_steady_complete["cycle_time"].to_numpy(dtype=float)
    mu_total = cycle_times.mean()
    var_total = ((cycle_times - mu_total) ** 2).mean()

    # -------- build MACHINE-LEVEL paths from running events (FIXED) --------
    df_run = df_steady[df_steady["status"] == "running"].copy()
    if df_run.empty:
        raise ValueError(f"No 'running' events for steady cases in {csv_path}")

    df_run = df_run.sort_values(["case_id", "timestamp", "activity", "resource"])

    def build_path(group: pd.DataFrame) -> str:
        nodes = []
        for _, row in group.iterrows():
            res = row.get("resource", None)
            # Node = resource name if available, else activity label
            if isinstance(res, str) and res != "":
                node = res
            else:
                node = str(row["activity"])
            nodes.append(node)
        # IMPORTANT: NO compression; every visit (loops due to QC) stays in the path
        return ">".join(nodes)

    paths = (
        df_run.groupby("case_id")
              .apply(build_path)
              .reset_index(name="path")
    )

    case_level = df_steady_complete[["case_id", "cycle_time", "start_time"]].merge(
        paths, on="case_id", how="left"
    )
    case_level = case_level.dropna(subset=["path"]).copy()
    N = len(case_level)
    if N == 0:
        raise ValueError(f"No cases with paths in {csv_path}")

    # -------- variance decomposition by path (now w.r.t. machine-level paths) --------
    path_stats = (
        case_level.groupby("path")["cycle_time"]
                  .agg(n="count", mean="mean", var=pop_var)
                  .reset_index()
    )

    N_check = path_stats["n"].sum()
    assert N_check == N

    weights = path_stats["n"] / N_check
    mu_from_paths = (path_stats["mean"] * path_stats["n"]).sum() / N_check

    Var_between = ((path_stats["mean"] - mu_from_paths) ** 2 * weights).sum()
    Var_within  = (path_stats["var"] * weights).sum()
    Var_total_check = Var_between + Var_within

    if var_total > 0:
        D_paths = Var_between / var_total
    else:
        D_paths = np.nan

    # -------- path entropy --------
    pi = path_stats["n"] / N_check
    H = -(pi * np.log(pi)).sum()

    K = path_stats.shape[0]
    if K <= 1:
        H_norm = 0.0
    else:
        H_max = np.log(K)
        H_norm = H / H_max

    print(f"  N_steady (with paths): {N}")
    print(f"  mu_total: {mu_total:.4f}, Var_total: {var_total:.4f}")
    print(f"  Var_between: {Var_between:.4f}, Var_within: {Var_within:.4f}")
    print(f"  Var_between + Var_within: {Var_total_check:.4f}")
    print(f"  K_paths: {K}, D_paths: {D_paths:.4f}, H_norm: {H_norm:.4f}")

    return {
        "n_steady_cases": N,
        "mu_total": mu_total,
        "var_total": var_total,
        "var_between": Var_between,
        "var_within": Var_within,
        "var_bw_plus_within": Var_total_check,
        "D_paths": D_paths,
        "H": H,
        "H_norm": H_norm,
        "K_paths": K,
        "warmup_frac": warmup_frac,
        "warmup_threshold": warmup_threshold,
        "t_max": t_max,
    }


rows = []
for style in styles:
    for hk in hets:
        fname = (
            f"FIFO_EXP_l{l_value}_{style}_{count_key}_"
            f"V{variant_count}_A{activity_total}_QC{int(qc_level*100)}_{hk}.csv"
        )
        csv_path = os.path.join(RESULTS_DIR, fname)
        if not os.path.exists(csv_path):
            print(f"WARNING: missing file, skipping: {csv_path}")
            continue

        stats = compute_descriptors_for_file(csv_path, warmup_frac=WARMUP_FRAC)
        stats.update({
            "style": style,
            "hetero": hk,
            "l": l_value,
            "count_preset": count_key,
            "variant_count": variant_count,
            "activity_total": activity_total,
            "qc_level": qc_level,
        })
        rows.append(stats)

summary_df = pd.DataFrame(rows)

print("\n=== Path-based descriptor summary (machine-level) ===")
print(summary_df[[
    "style", "hetero", "n_steady_cases",
    "mu_total", "var_total", "var_between", "var_within",
    "D_paths", "H_norm", "K_paths"
]])

out_path = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_level.csv")
summary_df.to_csv(out_path, index=False)
print(f"\nSaved summary to {out_path}")



=== Processing FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\2385334664.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6979
  mu_total: 69.4456, Var_total: 1068.8277
  Var_between: 473.2112, Var_within: 595.6165
  Var_between + Var_within: 1068.8277
  K_paths: 2632, D_paths: 0.4427, H_norm: 0.9590

=== Processing FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_mild_all.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\2385334664.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6899
  mu_total: 65.7569, Var_total: 961.9970
  Var_between: 427.2715, Var_within: 534.7255
  Var_between + Var_within: 961.9970
  K_paths: 2663, D_paths: 0.4442, H_norm: 0.9596

=== Processing FIFO_EXP_l0.28_hybrid30_C1_V18_A8_QC97_identical.csv ===


...

C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\2385334664.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6984
  mu_total: 87.5765, Var_total: 1974.2524
  Var_between: 851.5014, Var_within: 1122.7509
  Var_between + Var_within: 1974.2524
  K_paths: 2641, D_paths: 0.4313, H_norm: 0.9599

=== Processing FIFO_EXP_l0.28_dedicated_C1_V18_A8_QC97_identical.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\2385334664.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6919
  mu_total: 92.7148, Var_total: 2297.2068
  Var_between: 667.1368, Var_within: 1630.0700
  Var_between + Var_within: 2297.2068
  K_paths: 1465, D_paths: 0.2904, H_norm: 0.8885

=== Processing FIFO_EXP_l0.28_dedicated_C1_V18_A8_QC97_mild_all.csv ===
  N_steady (with paths): 6983
  mu_total: 99.7021, Var_total: 2385.8943
  Var_between: 706.3461, Var_within: 1679.5482
  Var_between + Var_within: 2385.8943
  K_paths: 1483, D_paths: 0.2961, H_norm: 0.8875

=== Path-based descriptor summary (machine-level) ===
       style     hetero  n_steady_cases   mu_total    var_total  var_between  \
0     pooled  identical            6979  69.445600  1068.827746   473.211235   
1     pooled   mild_all            6899  65.756911   961.996963   427.271502   
2   hybrid30  identical            6922  77.130785  1260.504645   527.251899   
3   hybrid30   mild_all            6984  87.576527  1974.252354   851.501411   
4  dedicated  identical            6919  92.714835  2297.206

C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\2385334664.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


In [4]:
#!/usr/bin/env python

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"
RESULTS_DIR = os.path.join(OUT_ROOT, "results")

l_value       = 0.28
styles        = ["pooled", "hybrid30", "dedicated"]
count_key     = "C1"
variant_count = 18
activity_total = 8
qc_level      = 0.97
hets          = ["identical", "mild_all"]

# Use the same warm-up fraction as before
WARMUP_FRAC = 0.17


def activity_to_station(label: str) -> str:
    """Map detailed activity labels to station-level buckets."""
    if label.startswith("MOULDING"):
        return "MOULD"
    if label.startswith("ASSEMBLY_1"):
        return "A1"
    if label.startswith("ASSEMBLY_2"):
        return "A2"
    if label.startswith("SORTING"):
        return "SORT"
    if label.startswith("PACKAGING"):
        return "PACK"
    if label.startswith("INSPECTION"):
        return "INSP"
    return label


def pop_var(x: pd.Series) -> float:
    x = x.to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    mu = x.mean()
    return ((x - mu) ** 2).mean()


def compute_descriptors_for_file(csv_path: str, warmup_frac: float = 0.17) -> dict:
    print(f"\n=== Processing {os.path.basename(csv_path)} ===")
    df = pd.read_csv(csv_path)

    # -------- warm-up selection on cases --------
    df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
    df_start = df_start.rename(columns={"timestamp": "start_time"})

    df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
    if df_complete.empty:
        raise ValueError(f"No COMPLETE END events in {csv_path}")

    df_complete = df_complete.merge(df_start, on="case_id", how="left")

    t_max = df["timestamp"].max()
    warmup_threshold = warmup_frac * t_max
    df_steady_complete = df_complete[df_complete["start_time"] >= warmup_threshold].copy()
    if df_steady_complete.empty:
        raise ValueError(f"No cases left after warm-up in {csv_path}")

    steady_case_ids = df_steady_complete["case_id"].unique()
    df_steady = df[df["case_id"].isin(steady_case_ids)].copy()

    # overall cycle-time stats (for consistency)
    cycle_times = df_steady_complete["cycle_time"].to_numpy(dtype=float)
    mu_total = cycle_times.mean()
    var_total = ((cycle_times - mu_total) ** 2).mean()

    # -------- build paths from running events --------
    df_run = df_steady[df_steady["status"] == "running"].copy()
    if df_run.empty:
        raise ValueError(f"No 'running' events for steady cases in {csv_path}")

    df_run["station"] = df_run["activity"].astype(str).map(activity_to_station)
    df_run = df_run.sort_values(["case_id", "timestamp", "activity"])

    def compress_stations(stations):
        out = []
        prev = None
        for s in stations:
            if s != prev:
                out.append(s)
                prev = s
        return ">".join(out)

    paths = (
        df_run.groupby("case_id")["station"]
              .apply(compress_stations)   # or ">".join if you don't want compression
              .reset_index()
              .rename(columns={"station": "path"})
    )

    case_level = df_steady_complete[["case_id", "cycle_time", "start_time"]].merge(
        paths, on="case_id", how="left"
    )
    case_level = case_level.dropna(subset=["path"]).copy()
    N = len(case_level)
    if N == 0:
        raise ValueError(f"No cases with paths in {csv_path}")

    # -------- variance decomposition by path --------
    path_stats = (
        case_level.groupby("path")["cycle_time"]
                  .agg(n="count", mean="mean", var=pop_var)
                  .reset_index()
    )

    N_check = path_stats["n"].sum()
    assert N_check == N

    weights = path_stats["n"] / N_check
    mu_from_paths = (path_stats["mean"] * path_stats["n"]).sum() / N_check

    Var_between = ((path_stats["mean"] - mu_from_paths) ** 2 * weights).sum()
    Var_within  = (path_stats["var"] * weights).sum()
    Var_total_check = Var_between + Var_within

    if var_total > 0:
        D_paths = Var_between / var_total
    else:
        D_paths = np.nan

    # -------- path entropy --------
    pi = path_stats["n"] / N_check
    H = -(pi * np.log(pi)).sum()

    K = path_stats.shape[0]
    if K <= 1:
        H_norm = 0.0
    else:
        H_max = np.log(K)
        H_norm = H / H_max

    print(f"  N_steady (with paths): {N}")
    print(f"  mu_total: {mu_total:.4f}, Var_total: {var_total:.4f}")
    print(f"  Var_between: {Var_between:.4f}, Var_within: {Var_within:.4f}")
    print(f"  Var_between + Var_within: {Var_total_check:.4f}")
    print(f"  D_paths: {D_paths:.4f}, H_norm: {H_norm:.4f}")

    return {
        "n_steady_cases": N,
        "mu_total": mu_total,
        "var_total": var_total,
        "var_between": Var_between,
        "var_within": Var_within,
        "var_bw_plus_within": Var_total_check,
        "D_paths": D_paths,
        "H": H,
        "H_norm": H_norm,
        "K_paths": K,
        "warmup_frac": warmup_frac,
        "warmup_threshold": warmup_threshold,
        "t_max": t_max,
    }


rows = []
for style in styles:
    for hk in hets:
        fname = (
            f"FIFO_EXP_l{l_value}_{style}_{count_key}_"
            f"V{variant_count}_A{activity_total}_QC{int(qc_level*100)}_{hk}.csv"
        )
        csv_path = os.path.join(RESULTS_DIR, fname)
        if not os.path.exists(csv_path):
            print(f"WARNING: missing file, skipping: {csv_path}")
            continue

        stats = compute_descriptors_for_file(csv_path, warmup_frac=WARMUP_FRAC)
        stats.update({
            "style": style,
            "hetero": hk,
            "l": l_value,
            "count_preset": count_key,
            "variant_count": variant_count,
            "activity_total": activity_total,
            "qc_level": qc_level,
        })
        rows.append(stats)

summary_df = pd.DataFrame(rows)

print("\n=== Path-based descriptor summary ===")
print(summary_df[[
    "style", "hetero", "n_steady_cases",
    "mu_total", "var_total", "var_between", "var_within",
    "D_paths", "H_norm", "K_paths"
]])

out_path = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_summary.csv")
summary_df.to_csv(out_path, index=False)
print(f"\nSaved summary to {out_path}")



=== Processing FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_identical.csv ===
  N_steady (with paths): 6979
  mu_total: 69.4456, Var_total: 1068.8277
  Var_between: 0.0000, Var_within: 1068.8277
  Var_between + Var_within: 1068.8277
  D_paths: 0.0000, H_norm: 0.0000

=== Processing FIFO_EXP_l0.28_pooled_C1_V18_A8_QC97_mild_all.csv ===
  N_steady (with paths): 6899
  mu_total: 65.7569, Var_total: 961.9970
  Var_between: 0.0000, Var_within: 961.9970
  Var_between + Var_within: 961.9970
  D_paths: 0.0000, H_norm: 0.0000

=== Processing FIFO_EXP_l0.28_hybrid30_C1_V18_A8_QC97_identical.csv ===
  N_steady (with paths): 6922
  mu_total: 77.1308, Var_total: 1260.5046
  Var_between: 0.0000, Var_within: 1260.5046
  Var_between + Var_within: 1260.5046
  D_paths: 0.0000, H_norm: 0.0000

=== Processing FIFO_EXP_l0.28_hybrid30_C1_V18_A8_QC97_mild_all.csv ===
  N_steady (with paths): 6984
  mu_total: 87.5765, Var_total: 1974.2524
  Var_between: 0.0000, Var_within: 1974.2524
  Var_between + Var_within: 1974.

In [8]:

import os
import pandas as pd
import numpy as np

RUN_TAG  = "251110"
OUT_ROOT = f"out/{RUN_TAG}"
RESULTS_DIR = os.path.join(OUT_ROOT, "results")

# === match fifo.py experiment menu ===
l_values        = [0.28]                        # ARRIVAL_RATES in config
styles          = ["pooled", "hybrid30", "dedicated"]
count_keys      = ["C1"]                        # same as counts in fifo
variant_counts  = [6]                       # variants
activity_totals = [5, 8, 12]                        # acts
qc_levels       = [0.97, 0.99]                        # qc_lvls
hets            = ["identical", "mild_all", "strong_all"]     # hets

# Same warm-up fraction
WARMUP_FRAC = 0.17

def pop_var(x: pd.Series) -> float:
    x = x.to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    mu = x.mean()
    return ((x - mu) ** 2).mean()


def compute_descriptors_for_file(csv_path: str, warmup_frac: float = 0.17) -> dict:
    print(f"\n=== Processing {os.path.basename(csv_path)} ===")
    df = pd.read_csv(csv_path)

    # -------- warm-up selection on cases (KEEP THIS) --------
    df_start = df[df["status"] == "START"][["case_id", "timestamp"]].copy()
    df_start = df_start.rename(columns={"timestamp": "start_time"})

    df_complete = df[(df["status"] == "COMPLETE") & (df["activity"] == "END")].copy()
    if df_complete.empty:
        raise ValueError(f"No COMPLETE END events in {csv_path}")

    df_complete = df_complete.merge(df_start, on="case_id", how="left")

    t_max = df["timestamp"].max()
    warmup_threshold = warmup_frac * t_max
    df_steady_complete = df_complete[df_complete["start_time"] >= warmup_threshold].copy()
    if df_steady_complete.empty:
        raise ValueError(f"No cases left after warm-up in {csv_path}")

    steady_case_ids = df_steady_complete["case_id"].unique()
    df_steady = df[df["case_id"].isin(steady_case_ids)].copy()

    # -------- overall cycle-time stats (KEEP THIS) --------
    cycle_times = df_steady_complete["cycle_time"].to_numpy(dtype=float)
    mu_total = cycle_times.mean()
    var_total = ((cycle_times - mu_total) ** 2).mean()

    # -------- build MACHINE-LEVEL paths from running events --------
    df_run = df_steady[df_steady["status"] == "running"].copy()
    if df_run.empty:
        raise ValueError(f"No 'running' events for steady cases in {csv_path}")

    df_run = df_run.sort_values(["case_id", "timestamp", "activity", "resource"])

    def build_path(group: pd.DataFrame) -> str:
        nodes = []
        for _, row in group.iterrows():
            res = row.get("resource", None)
            node = res if isinstance(res, str) and res != "" else str(row["activity"])
            nodes.append(node)
        return ">".join(nodes)

    paths = (
        df_run.groupby("case_id", group_keys=False)  # group_keys=False to avoid FutureWarning
              .apply(build_path)
              .reset_index(name="path")
    )

    case_level = df_steady_complete[["case_id", "cycle_time", "start_time"]].merge(
        paths, on="case_id", how="left"
    )
    case_level = case_level.dropna(subset=["path"]).copy()
    N = len(case_level)
    if N == 0:
        raise ValueError(f"No cases with paths in {csv_path}")

    # -------- variance decomposition by path --------
    path_stats = (
        case_level.groupby("path")["cycle_time"]
                  .agg(n="count", mean="mean", var=pop_var)
                  .reset_index()
    )

    N_check = path_stats["n"].sum()
    assert N_check == N

    weights = path_stats["n"] / N_check
    mu_from_paths = (path_stats["mean"] * path_stats["n"]).sum() / N_check

    Var_between = ((path_stats["mean"] - mu_from_paths) ** 2 * weights).sum()
    Var_within  = (path_stats["var"] * weights).sum()
    Var_total_check = Var_between + Var_within

    if var_total > 0:
        D_paths = Var_between / var_total
    else:
        D_paths = np.nan

    # -------- path entropy --------
    pi = path_stats["n"] / N_check
    H = -(pi * np.log(pi)).sum()

    K = path_stats.shape[0]
    if K <= 1:
        H_norm = 0.0
    else:
        H_max = np.log(K)
        H_norm = H / H_max

    print(f"  N_steady (with paths): {N}")
    print(f"  mu_total: {mu_total:.4f}, Var_total: {var_total:.4f}")
    print(f"  Var_between: {Var_between:.4f}, Var_within: {Var_within:.4f}")
    print(f"  Var_between + Var_within: {Var_total_check:.4f}")
    print(f"  K_paths: {K}, D_paths: {D_paths:.4f}, H_norm: {H_norm:.4f}")

    return {
        "n_steady_cases": N,
        "mu_total": mu_total,
        "var_total": var_total,
        "var_between": Var_between,
        "var_within": Var_within,
        "var_bw_plus_within": Var_total_check,
        "D_paths": D_paths,
        "H": H,
        "H_norm": H_norm,
        "K_paths": K,
        "warmup_frac": warmup_frac,
        "warmup_threshold": warmup_threshold,
        "t_max": t_max,
    }


# ---------- outer loop over all experiment files ----------
rows = []
for l_value in l_values:
    for style in styles:
        for count_key in count_keys:
            for variant_count in variant_counts:
                for activity_total in activity_totals:
                    for qc_level in qc_levels:
                        for hk in hets:
                            fname = (
                                f"FIFO_EXP_l{l_value}_{style}_{count_key}_"
                                f"V{variant_count}_A{activity_total}_QC{int(qc_level*100)}_{hk}.csv"
                            )
                            csv_path = os.path.join(RESULTS_DIR, fname)
                            if not os.path.exists(csv_path):
                                print(f"WARNING: missing file, skipping: {csv_path}")
                                continue

                            stats = compute_descriptors_for_file(csv_path, warmup_frac=WARMUP_FRAC)
                            stats.update({
                                "l": l_value,
                                "style": style,
                                "count_preset": count_key,
                                "variant_count": variant_count,
                                "activity_total": activity_total,
                                "qc_level": qc_level,
                                "hetero": hk,
                            })
                            rows.append(stats)

summary_df = pd.DataFrame(rows)

print("\n=== Path-based descriptor summary (machine-level) ===")
print(summary_df[[
    "l", "style", "hetero", "variant_count", "activity_total",
    "n_steady_cases", "mu_total", "var_total",
    "var_between", "var_within", "D_paths", "H_norm", "K_paths"
]])

out_path = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_level.csv")
summary_df.to_csv(out_path, index=False)
print(f"\nSaved summary to {out_path}")



=== Processing FIFO_EXP_l0.28_pooled_C1_V6_A5_QC97_identical.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\1530796084.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6937
  mu_total: 70.9119, Var_total: 1580.3707
  Var_between: 328.5892, Var_within: 1251.7815
  Var_between + Var_within: 1580.3707
  K_paths: 908, D_paths: 0.2079, H_norm: 0.8241

=== Processing FIFO_EXP_l0.28_pooled_C1_V6_A5_QC97_mild_all.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\1530796084.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6950
  mu_total: 56.7799, Var_total: 734.4128
  Var_between: 128.4282, Var_within: 605.9846
  Var_between + Var_within: 734.4128
  K_paths: 903, D_paths: 0.1749, H_norm: 0.8233

=== Processing FIFO_EXP_l0.28_pooled_C1_V6_A5_QC97_strong_all.csv ===


...

C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\1530796084.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6854
  mu_total: 122.1227, Var_total: 5336.4809
  Var_between: 1852.9321, Var_within: 3483.5488
  Var_between + Var_within: 5336.4809
  K_paths: 1077, D_paths: 0.3472, H_norm: 0.8842

=== Processing FIFO_EXP_l0.28_dedicated_C1_V6_A12_QC99_strong_all.csv ===


C:\Users\990215322\AppData\Local\Temp\ipykernel_27400\1530796084.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_path)


  N_steady (with paths): 6671
  mu_total: 220.4169, Var_total: 60099.2051
  Var_between: 26211.1419, Var_within: 33888.0632
  Var_between + Var_within: 60099.2051
  K_paths: 1032, D_paths: 0.4361, H_norm: 0.8863

=== Path-based descriptor summary (machine-level) ===
       l      style      hetero  variant_count  activity_total  \
0   0.28     pooled   identical              6               5   
1   0.28     pooled    mild_all              6               5   
2   0.28     pooled  strong_all              6               5   
3   0.28     pooled   identical              6               5   
4   0.28     pooled    mild_all              6               5   
5   0.28     pooled  strong_all              6               5   
6   0.28     pooled   identical              6               8   
7   0.28     pooled    mild_all              6               8   
8   0.28     pooled  strong_all              6               8   
9   0.28     pooled   identical              6               8   
10  0.2

PermissionError: [Errno 13] Permission denied: 'out/251110\\results\\FIFO_EXP_path_descriptors_machine_level.csv'